# 🏗️ Data Mesh Hub — Overview & Infrastructure Setup

## What is Data Mesh?
Data Mesh is a **decentralized** approach to data architecture where **domain teams own their data as products**. Each domain is responsible for producing high-quality, discoverable data products within **their own workspace and catalog**.

## Architecture: Hub & Spoke (Workspace-per-Domain)

| Workspace | Catalog | Domain | Data Product | Repo |
|---|---|---|---|---|
| **dbx-dps-raise-dev** | `adoit_product` | Enterprise Architecture | Application Landscape | `dbx-dps-raise` |
| **dbx-dps-snow-dev** | `snow_product` | IT Service Management | Service Health | `dbx-dp-snow` |
| **This workspace (Hub)** | `data_mesh_hub` | Cross-Domain | Application Risk Profile | `data-mesh-demo` |

## This Notebook
Sets up the **Hub** infrastructure only:
- `data_mesh_hub.cross_domain` — Cross-domain federated products
- `data_mesh_hub.platform` — Quality monitoring, product registry, contracts, observability

> **Domain catalogs** (`adoit_product`, `snow_product`) are created and managed by their respective domain workspaces. The hub reads their gold tables via Unity Catalog's shared metastore.

## Step 1: Create Hub Catalog & Schemas
The Hub provides cross-domain federation and platform governance. Domain catalogs are managed independently.

In [0]:
%sql
-- ============================================================
-- HUB CATALOG: data_mesh_hub (Platform Team owns this)
-- Domain catalogs are created by their respective workspaces
-- ============================================================
CREATE CATALOG IF NOT EXISTS data_mesh_hub 
MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/data_mesh_hub'
COMMENT 'Data Mesh Hub Catalog | Owner: Data Platform Team | Cross-Domain federation and governance';

CREATE SCHEMA IF NOT EXISTS data_mesh_hub.cross_domain COMMENT 'Cross-domain data products | Federated from adoit_product and snow_product';
CREATE SCHEMA IF NOT EXISTS data_mesh_hub.platform     COMMENT 'Platform governance tables | Registry, contracts, observability';

In [0]:
# Clean up dev-prefixed schemas (bundle development mode artifacts)
dev_schemas = [
    'adoit_product.dev_data_mesh_cicd_bronze',
    'adoit_product.dev_data_mesh_cicd_silver',
    'adoit_product.dev_data_mesh_cicd_gold',
    'adoit_product.default',
    'snow_product.dev_data_mesh_cicd_bronze',
    'snow_product.dev_data_mesh_cicd_silver',
    'snow_product.dev_data_mesh_cicd_gold',
    'snow_product.default',
    'data_mesh_hub.dev_data_mesh_cicd_cross_domain',
    'data_mesh_hub.dev_data_mesh_cicd_platform',
    'data_mesh_hub.default',
]
for s in dev_schemas:
    try:
        spark.sql(f'DROP SCHEMA IF EXISTS {s} CASCADE')
        print(f'  Dropped {s}')
    except Exception as e:
        pass  # Schema may not exist
print('✓ Dev schema cleanup complete')


In [0]:
%sql
-- ============================================================
-- Platform tables — schemas match EXACT column usage across notebooks
-- ============================================================

-- Data Product Registry (14 cols: seed INSERT + NB07 reads table_name + NB07 updates)
CREATE OR REPLACE TABLE data_mesh_hub.platform.data_product_registry (
  product_id STRING, product_name STRING, domain STRING, owner_group STRING,
  owner_contact STRING, table_name STRING, status STRING,
  sla_freshness_hours INT, quality_score DOUBLE, freshness_status STRING,
  consumer_count INT, consumers STRING, created_at TIMESTAMP, updated_at TIMESTAMP,
  last_refreshed_at TIMESTAMP, row_count LONG
);

-- Data Contracts (13 cols: matches seed INSERT + NB07/NB08 reads)
CREATE OR REPLACE TABLE data_mesh_hub.platform.data_contracts (
  contract_id STRING, product_id STRING, product_name STRING, producer_group STRING,
  consumer_group STRING, contract_status STRING, agreed_sla_hours INT,
  quality_threshold_pct DOUBLE, schema_version STRING, retention_days INT,
  escalation_contact STRING, escalation_channel STRING, created_at TIMESTAMP
);

-- Data Quality Log (12 cols: matches NB04 INSERT exactly)
CREATE OR REPLACE TABLE data_mesh_hub.platform.data_quality_log (
  check_id STRING, check_timestamp TIMESTAMP, data_product STRING,
  domain STRING, table_reference STRING, check_name STRING,
  check_type STRING, description STRING, passed BOOLEAN,
  expected_value STRING, actual_value STRING, severity STRING
);

-- Product Observability (16 cols: matches NB07 Cell 12 INSERT VALUES)
CREATE OR REPLACE TABLE data_mesh_hub.platform.product_observability (
  obs_id STRING, product_id STRING, product_name STRING, check_timestamp TIMESTAMP,
  row_count LONG, row_count_change LONG, freshness_hours DOUBLE, sla_met BOOLEAN,
  quality_checks_total INT, quality_checks_passed INT, quality_score DOUBLE,
  column_count INT, schema_drift BOOLEAN, active_contracts INT,
  consumer_count INT, status STRING
);

-- Domain Maturity Scorecard (14 cols: matches seed INSERT)
CREATE OR REPLACE TABLE data_mesh_hub.platform.domain_maturity_scorecard (
  domain STRING, maturity_level STRING, overall_maturity_score INT,
  data_quality_score INT, accessibility_score INT, documentation_score INT,
  interoperability_score INT, governance_score INT, security_score INT,
  observability_score INT, self_serve_score INT, product_count INT,
  consumer_count INT, uptime_pct DOUBLE
);

-- Cross-domain product table (22 cols: matches NB03 MERGE source)
CREATE OR REPLACE TABLE data_mesh_hub.cross_domain.application_risk_profile (
  app_id STRING, app_name STRING, app_owner STRING, department STRING,
  criticality STRING, lifecycle_status STRING, technology_stack STRING,
  is_cloud_hosted BOOLEAN, app_age_years DOUBLE, capability_count INT,
  total_incidents INT, open_incidents INT, critical_incidents INT,
  sla_breaches INT, sla_compliance_pct DOUBLE, avg_resolution_hours DOUBLE,
  total_changes INT, change_success_rate_pct DOUBLE, service_risk_score STRING,
  composite_risk_score STRING, risk_factors STRING, product_generated_at TIMESTAMP
);

In [0]:
# ── Load platform seed data (inline SQL) ──
seed_sql = """
-- ============================================================
-- Platform Governance Seed Data
-- Run AFTER notebooks 00-04 have created the base tables
-- ============================================================

-- Data Product Registry
INSERT INTO data_mesh_hub.platform.data_product_registry 
(product_id, product_name, domain, owner_group, owner_contact, table_name, status, sla_freshness_hours, quality_score, freshness_status, consumer_count, consumers, created_at, updated_at)
VALUES
('DP-001', 'Application Landscape', 'Enterprise Architecture', 'ea-team', 'ea-team@company.com', 'adoit_product.gold.application_landscape', 'published', 4, 98.5, 'fresh', 10, 'cto-office, risk-committee', current_timestamp(), current_timestamp()),
('DP-002', 'Service Health', 'IT Service Management', 'itsm-team', 'itsm-team@company.com', 'snow_product.gold.service_health', 'published', 2, 95.0, 'fresh', 9, 'cto-office, risk-committee', current_timestamp(), current_timestamp()),
('DP-003', 'Application Risk Profile', 'Cross-Domain', 'data-platform-team', 'platform@company.com', 'data_mesh_hub.cross_domain.application_risk_profile', 'published', 6, 96.0, 'fresh', 10, 'cto-office, risk-committee, compliance', current_timestamp(), current_timestamp());

-- Data Contracts
INSERT INTO data_mesh_hub.platform.data_contracts VALUES
('DC-001', 'DP-001', 'Application Landscape', 'ea-team', 'cto-office', 'active', 4, 95.0, 'v1.0', 30, 'ea-lead@company.com', '#ea-data-products', current_timestamp()),
('DC-002', 'DP-002', 'Service Health', 'itsm-team', 'cto-office', 'active', 2, 90.0, 'v1.0', 14, 'itsm-lead@company.com', '#itsm-data-products', current_timestamp()),
('DC-003', 'DP-002', 'Service Health', 'itsm-team', 'risk-committee', 'active', 2, 90.0, 'v1.0', 14, 'itsm-lead@company.com', '#itsm-data-products', current_timestamp()),
('DC-004', 'DP-003', 'Application Risk Profile', 'data-platform-team', 'cto-office', 'active', 6, 90.0, 'v1.0', 30, 'platform-lead@company.com', '#data-mesh-platform', current_timestamp()),
('DC-005', 'DP-003', 'Application Risk Profile', 'data-platform-team', 'risk-committee', 'active', 6, 90.0, 'v1.0', 30, 'platform-lead@company.com', '#data-mesh-platform', current_timestamp());

-- Domain Maturity Scorecard
INSERT INTO data_mesh_hub.platform.domain_maturity_scorecard VALUES
('Enterprise Architecture', 'Advanced', 92, 95, 90, 98, 88, 95, 92, 90, 85, 2, 3, 99.5),
('IT Service Management', 'Established', 85, 90, 82, 95, 85, 88, 85, 80, 78, 3, 4, 98.0),
('Cross-Domain', 'Established', 88, 85, 88, 96, 90, 92, 90, 82, 80, 2, 5, 99.0);
"""

# Execute each statement
for stmt in seed_sql.split(';'):
    stmt = stmt.strip()
    if stmt and not stmt.startswith('--'):
        try:
            spark.sql(stmt)
        except Exception as e:
            print(f'Warning: {str(e)[:100]}')
print('Platform seed data loaded')


## Step 2: Tag Catalogs, Schemas & Tables for Discoverability
Unity Catalog **Tags** at every level — catalog, schema, and table — make assets discoverable via search and Catalog Explorer.

In [0]:
%sql
-- Tag CATALOGS with domain ownership and mesh role
-- ALTER CATALOG adoit_product SET TAGS ('domain' = 'Enterprise Architecture', 'team_owner' = 'EA Team', 'data_mesh_role' = 'Domain Catalog');
-- ALTER CATALOG snow_product  SET TAGS ('domain' = 'IT Service Management', 'team_owner' = 'ITSM Team', 'data_mesh_role' = 'Domain Catalog');
ALTER CATALOG data_mesh_hub SET TAGS ('domain' = 'Cross-Domain', 'team_owner' = 'Data Platform Team', 'data_mesh_role' = 'Hub Catalog');

-- Tag SCHEMAS with domain, owner, and tier
ALTER SCHEMA adoit_product.bronze SET TAGS ('domain' = 'Enterprise Architecture', 'team_owner' = 'EA Team', 'tier' = 'bronze');
ALTER SCHEMA adoit_product.silver SET TAGS ('domain' = 'Enterprise Architecture', 'team_owner' = 'EA Team', 'tier' = 'silver');
ALTER SCHEMA adoit_product.gold   SET TAGS ('domain' = 'Enterprise Architecture', 'team_owner' = 'EA Team', 'tier' = 'gold');
ALTER SCHEMA snow_product.bronze  SET TAGS ('domain' = 'IT Service Management', 'team_owner' = 'ITSM Team', 'tier' = 'bronze');
ALTER SCHEMA snow_product.silver  SET TAGS ('domain' = 'IT Service Management', 'team_owner' = 'ITSM Team', 'tier' = 'silver');
ALTER SCHEMA snow_product.gold    SET TAGS ('domain' = 'IT Service Management', 'team_owner' = 'ITSM Team', 'tier' = 'gold');
ALTER SCHEMA data_mesh_hub.cross_domain SET TAGS ('domain' = 'Cross-Domain', 'team_owner' = 'Data Platform Team', 'tier' = 'gold');
ALTER SCHEMA data_mesh_hub.platform     SET TAGS ('domain' = 'Platform', 'team_owner' = 'Data Platform Team', 'tier' = 'platform');

-- NOTE: Gold table tagging is done in 08_Advanced_Governance (after tables are created by pipeline notebooks)

## Step 3: Access Control Patterns
With **separate catalogs**, each domain team has **full autonomy** over their access controls:

```sql
-- ============================================================
-- EA TEAM: Full control of their adoit_product catalog
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG adoit_product TO `ea-team`;

-- Consumers get READ-ONLY on the gold data products
GRANT USE CATALOG  ON CATALOG adoit_product TO `cto-office`;
GRANT USE SCHEMA   ON SCHEMA  adoit_product.gold TO `cto-office`;
GRANT SELECT       ON TABLE   adoit_product.gold.application_landscape TO `cto-office`;

-- ============================================================
-- ITSM TEAM: Full control of their snow_product catalog
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG snow_product TO `itsm-team`;

-- Consumers get READ-ONLY on gold
GRANT USE CATALOG  ON CATALOG snow_product TO `risk-team`;
GRANT USE SCHEMA   ON SCHEMA  snow_product.gold TO `risk-team`;
GRANT SELECT       ON TABLE   snow_product.gold.service_health TO `risk-team`;

-- ============================================================
-- PLATFORM TEAM: Manages the hub, grants read to everyone
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG data_mesh_hub TO `data-platform-team`;

GRANT USE CATALOG ON CATALOG data_mesh_hub TO `cto-office`;
GRANT USE CATALOG ON CATALOG data_mesh_hub TO `risk-team`;
GRANT SELECT ON TABLE data_mesh_hub.cross_domain.application_risk_profile TO `cto-office`;
GRANT SELECT ON TABLE data_mesh_hub.cross_domain.application_risk_profile TO `risk-team`;
```

**Key difference from single-catalog**: The EA Team can manage `adoit_product` access without involving the Platform Team. True domain autonomy.

## Step 4: Inspect the Data Product Metadata
Data product properties are embedded in Unity Catalog, making them discoverable via `DESCRIBE EXTENDED` or the Catalog Explorer UI.

In [0]:
%sql
-- Verify catalogs and schemas created successfully
SHOW SCHEMAS IN adoit_product;

In [0]:
%sql
-- Verify SNOW schemas
SHOW SCHEMAS IN snow_product;

In [0]:
%sql
-- Verify Hub schemas
SHOW SCHEMAS IN data_mesh_hub;

## Next Steps
Run the domain-specific notebooks to see the full pipeline in action:
- **01_ADOIT_Data_Product_Pipeline** — Bronze→Silver→Gold within `adoit_product` catalog
- **02_SNOW_Data_Product_Pipeline** — Bronze→Silver→Gold within `snow_product` catalog
- **03_Cross_Domain_Data_Product** — Cross-catalog federated consumption in `data_mesh_hub`
- **04_Data_Quality_Monitoring** — Quality checks across all 3 catalogs

In [0]:
%sql
-- Grant read access to all workspace users on data mesh catalogs
GRANT USE CATALOG ON CATALOG adoit_product TO `account users`;
GRANT USE SCHEMA ON CATALOG adoit_product TO `account users`;
GRANT SELECT ON CATALOG adoit_product TO `account users`;

GRANT USE CATALOG ON CATALOG snow_product TO `account users`;
GRANT USE SCHEMA ON CATALOG snow_product TO `account users`;
GRANT SELECT ON CATALOG snow_product TO `account users`;

GRANT USE CATALOG ON CATALOG data_mesh_hub TO `account users`;
GRANT USE SCHEMA ON CATALOG data_mesh_hub TO `account users`;
GRANT SELECT ON CATALOG data_mesh_hub TO `account users`;
